# Milestone M10: System Benchmarking & Computational Profiling Notebook

**Objective**: Measure the computational latency, Real-Time Factor (RTF), Peak RAM, and Peak GPU VRAM across each individual module stage and the end-to-end speech engineering pipeline over scaling audio durations (15s, 30s, 60s, 120s).

### Standardized Computational Metrics:
1. **Processing Latency ($T_{\text{proc}}$)**: High-resolution wall-clock execution time via `time.perf_counter()`.
2. **Real-Time Factor (RTF)**: $\text{RTF} = \frac{T_{\text{proc}}}{T_{\text{audio}}}$. Values $< 1.0$ indicate faster-than-realtime operation.
3. **Throughput Ratio**: $\text{Throughput} = \frac{1}{\text{RTF}}$ (audio seconds processed per second of compute).
4. **Peak System Memory (RAM)**: Monitored via `psutil` RSS and `tracemalloc`.
5. **Peak Accelerator VRAM**: Monitored via `torch.cuda` / `torch.mps` APIs.

## Cell 1: Environment & Hardware Inspector
Inspects host CPU, available physical and virtual RAM, GPU acceleration, and PyTorch runtime.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.benchmarking.profiler import get_hardware_specs

hw_specs = get_hardware_specs()
print("=" * 70)
print("           SYSTEM HARDWARE & RUNTIME SPECIFICATIONS")
print("=" * 70)
for k, v in hw_specs.items():
    print(f"{k:<25}: {v}")
print("=" * 70)

## Cell 2: Benchmark Harness Utilities & Profiler Setup
Initializes `SystemProfiler` and prepares standardized audio fixtures (15s, 30s, 60s, 120s).

In [ ]:
from src.benchmarking.profiler import SystemProfiler
from src.pipeline import SpeechEngineeringPipeline
from scripts.run_system_benchmark import prepare_benchmark_audio_fixtures

# Prepare standardized audio clips
fixtures = prepare_benchmark_audio_fixtures(PROJECT_ROOT)
print(f"Audio fixtures prepared: {list(fixtures.keys())} seconds")

# Instantiate Master Pipeline Orchestrator and Profiler
profiler = SystemProfiler(warmup=True)
pipeline = SpeechEngineeringPipeline(mock_mode=True, enable_caching=False)
pipeline.initialize()
print("Pipeline orchestrator initialized successfully.")

## Cell 3: Isolated Stage Benchmarks (30-Second Standard Clip)
Profiles isolated latency, RTF, and RAM consumption for each individual module stage.

In [ ]:
from scripts.run_system_benchmark import run_isolated_stage_benchmarks
import pandas as pd

test_30s = fixtures[30]
isolated_results = run_isolated_stage_benchmarks(
    pipeline=pipeline,
    profiler=profiler,
    test_audio_path=test_30s,
    audio_duration=30.0,
)

records = []
for stage, res in isolated_results.items():
    records.append({
        "Stage": stage,
        "Latency (s)": round(res.elapsed_seconds, 3),
        "RTF": res.rtf,
        "Speedup": f"{res.throughput_ratio}x",
        "Peak RAM (MB)": res.ram_peak_mb,
        "Peak VRAM (MB)": res.vram_peak_mb,
    })

df_isolated = pd.DataFrame(records)
df_isolated

## Cell 4: Duration Scaling & Algorithmic Complexity Analysis
Evaluates computational scaling across audio durations (15s, 30s, 60s, 120s).

In [ ]:
from scripts.run_system_benchmark import run_duration_scaling_benchmark
from src.benchmarking.profiler import plot_rtf_scaling_curves

durations, stage_rtfs, scaling_matrix = run_duration_scaling_benchmark(
    pipeline=pipeline,
    profiler=profiler,
    fixture_paths=fixtures,
)

df_scaling = pd.DataFrame(scaling_matrix)
print("Duration Scaling Summary:")
display(df_scaling)

# Render RTF Scaling plot
out_rtf_png = PROJECT_ROOT / "outputs" / "benchmarks" / "rtf_scaling_curves.png"
plot_rtf_scaling_curves(durations, stage_rtfs, out_rtf_png)
print(f"RTF curves saved to: {out_rtf_png}")

## Cell 5: Memory Profile Waterfall
Visualizes system RAM and GPU memory usage across the pipeline lifecycle.

In [ ]:
from src.benchmarking.profiler import plot_memory_waterfall, plot_stage_latency_breakdown

stages = list(isolated_results.keys())
ram_vals = [res.ram_peak_mb for res in isolated_results.values()]

out_mem_png = PROJECT_ROOT / "outputs" / "benchmarks" / "memory_profile_waterfall.png"
plot_memory_waterfall(stages, ram_vals, out_mem_png)
print(f"Memory waterfall saved to: {out_mem_png}")

# Stacked latency breakdown
latencies = {k: v.elapsed_seconds for k, v in isolated_results.items() if k != "End-to-End Pipeline"}
out_lat_png = PROJECT_ROOT / "outputs" / "benchmarks" / "stage_latency_breakdown.png"
plot_stage_latency_breakdown(latencies, out_lat_png)
print(f"Latency breakdown chart saved to: {out_lat_png}")

## Cell 6: Metric Export & Official Benchmark Serialization
Exports structured metrics to `outputs/benchmarks/benchmark_summary.json`.

In [ ]:
import json
from datetime import datetime, timezone

benchmark_export = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "hardware": hw_specs,
    "isolated_stages": {k: v.to_dict() for k, v in isolated_results.items()},
    "scaling_matrix": scaling_matrix,
}

export_path = PROJECT_ROOT / "outputs" / "benchmarks" / "benchmark_summary.json"
with open(export_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_export, f, indent=2)

print(f"[OK] Benchmark summary successfully serialized to: {export_path}")